# 🧹 NLP Preprocessing: YouTube Comments
Este notebook implementa um pipeline completo de limpeza e normalização para comentários de futebol, preparando os dados para análise de sentimento e modelagem de tópicos.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from unidecode import unidecode

# Downloads necessários do NLTK
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')

# 1. Carregar dados consolidadores
# Usaremos o arquivo parquet gerado na análise anterior
df = pd.read_parquet('bundesliga_2425_cazetv_chat.parquet')
print(f"Dataset carregado: {len(df):,} comentários.")

## 2. Dicionários de Expansão
Definimos contrações comuns e acrônimos específicos do domínio do futebol.

In [ ]:
contractions = {
    "can't": "cannot",
    "won't": "will not",
    "it's": "it is",
    "i'm": "i am",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are"
}

football_acronyms = {
    "motm": "man of the match",
    "var": "video assistant referee",
    "gg": "good game",
    "goat": "greatest of all time",
    "ucl": "uefa champions league",
    "gk": "goalkeeper",
    "cb": "centre back",
    "st": "striker"
}

## 3. Pipeline de Limpeza
Implementamos as funções de regex para as tarefas de limpeza solicitadas.

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    
    # 1. Lowercasing
    text = text.lower()
    
    # 2. Remover URLs e Menções (@user)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@[\w:]+', '', text)
    
    # 3. Remover caracteres especiais e emoticons (ASCII apenas)
    text = unidecode(text) # Remove acentos
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 4. Expandir Contrações e Acrônimos
    words = text.split()
    words = [contractions.get(w, w) for w in words]
    words = [football_acronyms.get(w, w) for w in words]
    
    # 5. Normalizar palavras intensificadas (ex: goooooal -> goal)
    # Regra: reduz caracteres repetidos mais de 2 vezes para apenas 2
    words = [re.sub(r'(.)\1{2,}', r'\1\1', w) for w in words]
    
    return " ".join(words)

print("Exemplo de limpeza:")
sample = "GOOOOOOAL!!! @HarryKane is the GOAT! Check this link: https://youtube.com it's amazing"
print(f"Original: {sample}")
print(f"Limpo:    {clean_text(sample)}")

## 4. Tokenização, Lemmatização e Filtragem
Reduzimos as palavras às suas raízes e descartamos comentários irrelevantes.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english') + stopwords.words('portuguese'))

def preprocess_pipeline(text):
    cleaned = clean_text(text)
    
    # 6. Tokenização
    tokens = nltk.word_tokenize(cleaned)
    
    # 7. Lemmatização (Removendo stopwords)
    processed = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    
    # 8. Token Count Filtering (Mínimo de 5 tokens)
    if len(processed) < 5: return None
    
    return " ".join(processed)

# Aplicar pipeline
print("Iniciando processamento em massa... (isso pode levar alguns minutos)")
df['mensagem_limpa'] = df['mensagem'].apply(preprocess_pipeline)

# Remover linhas que foram filtradas (None)
df_final = df.dropna(subset=['mensagem_limpa'])

print(f"\nProcessamento concluído!")
print(f"Comentários originais: {len(df):,}")
print(f"Comentários após limpeza e filtragem (>5 tokens): {len(df_final):,}")

## 5. Exportação dos Dados Limpos
Salvamos o dataset pronto para análise.

In [ ]:
output_file = 'bundesliga_chat_preprocessed.parquet'
df_final_export = df_final.copy()
for col in df_final_export.columns:
    df_final_export[col] = df_final_export[col].astype(object)
df_final_export.to_parquet(output_file, index=False, engine='fastparquet')

print(f"Dataset pré-processado salvo em: {output_file}")
print("Amostra dos dados limpos:")
df_final[['mensagem', 'mensagem_limpa']].head(10)